<a href="https://colab.research.google.com/github/stfnnnnnnn/karl-mangahas-flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stfnnnnnnn/1st-act/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

>I use the validated model score to rank pages by **which pages should be reviewed first**. A higher score does not mean that a page must be refreshed or that it will definitely decline. It means that, among pages meeting the Week 7 eligibility rules, the page has a higher estimated risk of later visibility decline based on historical search behavior available before the outcome window.

>Week 6 showed that the leakage-safe three-feature Random Forest could concentrate more declining pages near the top of one client-grouped holdout, but its overall discrimination remained weak. That result was promising enough to continue, but not strong enough to treat the Week 5 model as final. For Week 7, I therefore refine the development specification rather than simply reuse the earlier `imp_prev30`, `clk_prev30`, and `pos_prev30` model unchanged.

>The revised timeline uses **90 historical days split into three 30-day blocks**, followed by a separate 30-day outcome window. I use the historical blocks to describe recent level, earlier level, movement, click-through behavior, position movement, volatility, and evidence coverage. The outcome remains a later decline in impressions and is never included in the reviewer-facing feature set.

>I also narrow the population to pages that have enough evidence and **appear stable at the decision point**. An eligible page must have sufficient recent historical impressions, sufficient observed page history, client GSC tracking available before the historical feature window, and recent historical impressions within the declared stability band relative to the preceding historical block. This makes the Week 7 task closer to the original project question: identify pages worth reviewing **before an obvious decline is already underway**.

>Because Week 6 also showed that one favorable grouped holdout is not enough to establish a reliable ranking, this notebook compares the transparent baseline, Logistic Regression, Random Forest, and HistGradientBoosting using **client-grouped cross-validation**. Mean grouped-CV Average Precision is used as the predeclared model-selection metric, while Precision@20 and Precision@50 remain the main operational measures of limited review capacity. ROC-AUC and the decline base rate provide additional context.

>I retain a learned model only when the validation evidence supports the added complexity. In particular, the selected model should improve the review ranking relative to both the underlying decline base rate and the transparent baseline rather than merely producing the highest score among several weak alternatives.

### How I turn the ranking into actions

The following **review archetypes are transparent policy categories, not learned clusters**. They translate model rank and observable historical evidence into practical workflow states.

| Review archetype | What I see | Suggested action | Why |
|---|---|---|---|
| High-value risk | High model rank + meaningful historical visibility | **Review now** | A possible decline deserves earlier attention when the page already has substantial search exposure. |
| Position deterioration | High model rank + worsening historical position | **Diagnose ranking loss** | Historical position movement provides a reason to inspect whether the page is losing search visibility before choosing an intervention. |
| Click-capture opportunity | Meaningful impressions + weak historical CTR | **Review click capture** | The page receives search exposure but may be converting that visibility into relatively few clicks; title, snippet, and intent deserve inspection. |
| Limited-evidence risk | High model rank but weaker historical evidence or coverage | **Monitor / review cautiously** | A high score is less convincing when the supporting historical evidence is limited. |
| Lower-priority page | Does not meet a stronger review pattern | **Standard review / monitor** | The available model evidence does not justify urgent editorial work. |

>The queue uses evidence-based reason codes such as `high_model_rank`, `meaningful_visibility`, `historical_position_worsening`, `weak_click_capture`, and `limited_evidence`. These codes are **not diagnoses or causes**. They explain which observable historical conditions contributed to the page being surfaced and give the reviewer a starting point for investigation.

>The known outcome is retained only for retrospective model evaluation. It is excluded from the reviewer-facing action queue because the later outcome would not exist when the actual review decision is made.

### Decay and refresh insight

>The **FlyRank research paper examined in Week 6** reported observed relationships between freshness and performance, including differences across freshness bands and between recently refreshed and less-recently updated mature pages. Those findings are useful context for the action playbook because they suggest that freshness may be one factor worth checking when a page is surfaced for review.

>I do **not** treat those observations as evidence that refreshing a page will cause recovery. Refreshed and non-refreshed pages may already differ in important ways, and the Week 7 model itself does not estimate the causal effect of an editorial intervention. A high-ranked page therefore requires diagnosis before a reviewer decides whether the appropriate response is refresh, protection, expansion, consolidation, monitoring, or no change.

>The operational implication is deliberately narrow: **freshness can help inform what a human investigates after a page is prioritized, but neither freshness nor the model score determines what the human should change.**

In [22]:
%pip -q install duckdb huggingface_hub

import os, getpass, json, platform
import numpy as np
import pandas as pd
from pathlib import Path

import duckdb
import sklearn

from IPython.display import display

from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


RANDOM_SEED = 42

MIN_RECENT_IMPRESSIONS = 100
MIN_HISTORY_DAYS = 60
STABILITY_LOWER = 0.80
STABILITY_UPPER = 1.20
DECLINE_RATIO = 0.80


HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("HF token: ")


con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")


REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/**/*.parquet'"
    f")"
)

CLIENTS = (
    f"read_parquet("
    f"'{REL}/dim_clients.parquet'"
    f")"
)


DEVELOPMENT_END = pd.Timestamp(
    con.sql(f"""
        SELECT MAX(report_date)
        FROM {FACT}
        WHERE month = '2026-03'
    """).fetchone()[0]
)

DECISION_DATE = DEVELOPMENT_END - pd.Timedelta(days=30)
HISTORICAL_FEATURE_START = DEVELOPMENT_END - pd.Timedelta(days=120)


features = con.sql(f"""
WITH bounds AS (
    SELECT
        MAX(report_date) AS end_d
    FROM {FACT}
    WHERE month = '2026-03'
),

windowed AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 120 DAY
                 AND f.report_date <= b.end_d - INTERVAL 90 DAY
                THEN f.gsc_impressions
                ELSE 0
            END
        ) AS imp_early30,

        SUM(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 90 DAY
                 AND f.report_date <= b.end_d - INTERVAL 60 DAY
                THEN f.gsc_impressions
                ELSE 0
            END
        ) AS imp_middle30,

        SUM(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                 AND f.report_date <= b.end_d - INTERVAL 30 DAY
                THEN f.gsc_impressions
                ELSE 0
            END
        ) AS imp_recent30,

        SUM(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                 AND f.report_date <= b.end_d
                THEN f.gsc_impressions
                ELSE 0
            END
        ) AS imp_outcome30,

        SUM(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 120 DAY
                 AND f.report_date <= b.end_d - INTERVAL 90 DAY
                THEN f.gsc_clicks
                ELSE 0
            END
        ) AS clk_early30,

        SUM(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 90 DAY
                 AND f.report_date <= b.end_d - INTERVAL 60 DAY
                THEN f.gsc_clicks
                ELSE 0
            END
        ) AS clk_middle30,

        SUM(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                 AND f.report_date <= b.end_d - INTERVAL 30 DAY
                THEN f.gsc_clicks
                ELSE 0
            END
        ) AS clk_recent30,

        AVG(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 120 DAY
                 AND f.report_date <= b.end_d - INTERVAL 90 DAY
                THEN f.gsc_avg_position
            END
        ) AS pos_early30,

        AVG(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 90 DAY
                 AND f.report_date <= b.end_d - INTERVAL 60 DAY
                THEN f.gsc_avg_position
            END
        ) AS pos_middle30,

        AVG(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                 AND f.report_date <= b.end_d - INTERVAL 30 DAY
                THEN f.gsc_avg_position
            END
        ) AS pos_recent30,

        COUNT(
            DISTINCT CASE
                WHEN f.report_date > b.end_d - INTERVAL 120 DAY
                 AND f.report_date <= b.end_d - INTERVAL 30 DAY
                THEN f.report_date
            END
        ) AS observed_history_days_90,

        COUNT(
            DISTINCT CASE
                WHEN f.report_date > b.end_d - INTERVAL 120 DAY
                 AND f.report_date <= b.end_d - INTERVAL 30 DAY
                 AND f.gsc_impressions > 0
                THEN f.report_date
            END
        ) AS days_with_impressions_90,

        AVG(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 120 DAY
                 AND f.report_date <= b.end_d - INTERVAL 30 DAY
                THEN f.gsc_impressions
            END
        ) AS avg_daily_imp_90,

        STDDEV_SAMP(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 120 DAY
                 AND f.report_date <= b.end_d - INTERVAL 30 DAY
                THEN f.gsc_impressions
            END
        ) AS sd_daily_imp_90

    FROM {FACT} f, bounds b

    WHERE f.report_date > b.end_d - INTERVAL 120 DAY
      AND f.report_date <= b.end_d

    GROUP BY
        f.client_hash_id,
        f.content_hash_id
)

SELECT
    w.*,
    c.gsc_data_start

FROM windowed w

LEFT JOIN {CLIENTS} c
    ON w.client_hash_id = c.client_hash_id
""").df()


features["tracking_history_ok"] = (
    features["gsc_data_start"].notna()
    & (
        pd.to_datetime(features["gsc_data_start"])
        <= HISTORICAL_FEATURE_START
    )
)


features["imp_90d"] = (
    features["imp_early30"]
    + features["imp_middle30"]
    + features["imp_recent30"]
)


features["stable_ratio_recent_vs_middle"] = np.where(
    features["imp_middle30"] > 0,
    features["imp_recent30"] / features["imp_middle30"],
    np.nan,
)


features["imp_change_middle_vs_early"] = np.where(
    features["imp_early30"] > 0,
    (
        features["imp_middle30"]
        - features["imp_early30"]
    )
    / features["imp_early30"],
    np.nan,
)


features["imp_change_recent_vs_middle"] = np.where(
    features["imp_middle30"] > 0,
    (
        features["imp_recent30"]
        - features["imp_middle30"]
    )
    / features["imp_middle30"],
    np.nan,
)


features["ctr_early30"] = np.where(
    features["imp_early30"] > 0,
    features["clk_early30"] / features["imp_early30"],
    np.nan,
)


features["ctr_middle30"] = np.where(
    features["imp_middle30"] > 0,
    features["clk_middle30"] / features["imp_middle30"],
    np.nan,
)


features["ctr_recent30"] = np.where(
    features["imp_recent30"] > 0,
    features["clk_recent30"] / features["imp_recent30"],
    np.nan,
)


features["ctr_change_recent_vs_middle"] = (
    features["ctr_recent30"]
    - features["ctr_middle30"]
)


features["pos_change_recent_vs_middle"] = (
    features["pos_recent30"]
    - features["pos_middle30"]
)


features["recent_position_available"] = (
    features["pos_recent30"].notna()
).astype(int)


features["imp_cv_90"] = np.where(
    features["avg_daily_imp_90"] > 0,
    features["sd_daily_imp_90"]
    / features["avg_daily_imp_90"],
    np.nan,
)


features["eligible"] = (
    (features["imp_recent30"] >= MIN_RECENT_IMPRESSIONS)
    & features["tracking_history_ok"]
    & (
        features["observed_history_days_90"]
        >= MIN_HISTORY_DAYS
    )
    & np.isfinite(
        features["stable_ratio_recent_vs_middle"]
    )
    & (
        features["stable_ratio_recent_vs_middle"]
        >= STABILITY_LOWER
    )
    & (
        features["stable_ratio_recent_vs_middle"]
        <= STABILITY_UPPER
    )
)


features["is_declining"] = (
    features["imp_outcome30"]
    < DECLINE_RATIO * features["imp_recent30"]
).astype(int)


frame = (
    features.loc[features["eligible"]]
    .copy()
    .reset_index(drop=True)
)


FEATURE_COLS = [
    "imp_early30",
    "imp_middle30",
    "imp_recent30",
    "clk_recent30",
    "pos_recent30",
    "imp_change_middle_vs_early",
    "imp_change_recent_vs_middle",
    "ctr_middle30",
    "ctr_recent30",
    "ctr_change_recent_vs_middle",
    "pos_change_recent_vs_middle",
    "days_with_impressions_90",
    "imp_cv_90",
    "recent_position_available",
]


X = (
    frame[FEATURE_COLS]
    .replace([np.inf, -np.inf], np.nan)
    .copy()
)

y = frame["is_declining"].copy()
groups = frame["client_hash_id"].copy()


print("Development endpoint:", DEVELOPMENT_END.date())
print("Decision date:", DECISION_DATE.date())
print(
    "Historical feature start:",
    HISTORICAL_FEATURE_START.date(),
)

print()
print("Candidate pages:", f"{len(features):,}")
print("Eligible pages:", f"{len(frame):,}")
print(
    "Eligible clients:",
    frame["client_hash_id"].nunique(),
)
print(
    "Eligible decline base rate:",
    f"{frame['is_declining'].mean():.2%}",
)
print(
    "Tracking-history eligible:",
    f"{features['tracking_history_ok'].mean():.2%}",
)
print("Feature count:", len(FEATURE_COLS))
print("Features:", FEATURE_COLS)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Development endpoint: 2026-03-31
Decision date: 2026-03-01
Historical feature start: 2025-12-01

Candidate pages: 349,411
Eligible pages: 16,183
Eligible clients: 24
Eligible decline base rate: 28.58%
Tracking-history eligible: 80.83%
Feature count: 14
Features: ['imp_early30', 'imp_middle30', 'imp_recent30', 'clk_recent30', 'pos_recent30', 'imp_change_middle_vs_early', 'imp_change_recent_vs_middle', 'ctr_middle30', 'ctr_recent30', 'ctr_change_recent_vs_middle', 'pos_change_recent_vs_middle', 'days_with_impressions_90', 'imp_cv_90', 'recent_position_available']


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

>The intended users are **content managers and editors who cannot manually inspect every page**. The playbook helps allocate limited review capacity by ranking pseudonymized pages from higher to lower review priority. One row in the queue represents one content page that meets the Week 7 eligibility requirements.

>The queue is a **decision-support tool**, not an automated editorial system. A reviewer can start with the top 20 or top 50 pages, inspect the reason codes and available historical context, and then decide whether to investigate, act, monitor, or dismiss the recommendation. The model score determines **review order**; it does not determine the final editorial action.

>Unlike the earlier three-feature model, the Week 7 specification uses a longer historical view. The **90-day historical feature period is divided into three 30-day blocks**, allowing the model to represent not only recent search levels but also historical movement, position changes, click-through behavior, volatility, and evidence coverage before the later outcome window begins.

>The playbook also narrows the modeling population to pages that have enough evidence and **appear stable at the decision point**. Eligibility requires sufficient recent historical impressions, sufficient observed page history, client GSC tracking before the historical feature window, and recent historical impressions within the declared stability band relative to the preceding historical block. These requirements are intended to reduce low-evidence cases and avoid treating pages that are already obviously deteriorating as early-warning candidates.

>The review archetypes are **transparent policy categories, not learned clusters**. They translate model rank and observable historical signals into practical review states. The known outcome is used only for retrospective model evaluation and is excluded from the reviewer-facing action queue because it would not be available when the review decision is made.

### Limits

>The model only sees the **historical search behavior represented by the Week 7 feature set**. It does not know the page's full content quality, business importance, search intent, query mix, seasonality, competing pages, recent editorial decisions, SERP changes, or client-specific events. These missing factors can make a high-ranked recommendation wrong.

>The stability rule is also deliberately narrow. A page is considered “stable” because its recent historical impressions fall within the declared stability band relative to the preceding historical block. This does **not** mean that the page is healthy or stable across every search, content, or engagement signal.

>The March analysis remains a **development evaluation**. Client-grouped cross-validation gives a stronger view of how model performance changes across held-out clients, and a fixed client-grouped holdout provides a concrete queue for error analysis and playbook construction. However, neither is the same as a sealed future-month evaluation. The measured performance should therefore not be assumed to hold across every future month or client.

>The decline label is an observational outcome: impressions in the later 30-day outcome window fall below the declared proportion of recent historical impressions. It identifies **what happened, not why it happened**. A decline may reflect genuine content decay, but it may also reflect seasonality, consolidation, changing search demand, SERP changes, broader site movement, or noise.

>The label also does not show that editing a flagged page would reverse the decline. The safest claim is therefore:

>* **The model provides directional decision support for prioritizing which eligible pages appear worth reviewing first based on historical search behavior.**

>* It does not predict Google's algorithm, diagnose the cause of decline, or guarantee that a content change will improve performance.

### Cost and value

>The practical cost of a **high-ranked actual negative** is wasted review capacity or an unnecessary intervention on a page that does not later meet the observed decline outcome. The cost of a **low-ranked actual positive** is missing a page that later declines and therefore may not receive early investigation.

>These costs are not equal for every page. A high-visibility page can represent more value to protect, while a low-evidence recommendation carries more uncertainty. This is why the playbook combines model rank with evidence-based reason codes and conservative review states such as `review_now`, `diagnose`, and `monitor` rather than assigning the same action to every high-scored page.

>Because review capacity is limited, **Precision@20 and Precision@50** remain especially important to this use case. However, Week 6 showed why a strong result at one review depth or on one grouped split should not be interpreted by itself. Week 7 therefore also uses grouped cross-validation and Average Precision to evaluate whether ranking quality is more consistent across held-out clients.

>I do not convert these errors into invented financial values because the data does not establish the monetary return from an editorial intervention. Here, value is operational: **concentrate limited review capacity where the historical evidence is strongest while keeping consequential editorial actions under human control.**

In [23]:
def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(scores))
    order = np.argsort(-scores)[:k]

    return float(
        y_true[order].mean()
    )


def lift_at_k(y_true, scores, k):
    base_rate = float(
        np.mean(y_true)
    )

    if base_rate <= 0:
        return np.nan

    return (
        precision_at_k(
            y_true,
            scores,
            k,
        )
        / base_rate
    )


def metric_row(name, y_true, scores):
    return {
        "method": name,
        "base_rate": float(
            np.mean(y_true)
        ),
        "roc_auc": float(
            roc_auc_score(
                y_true,
                scores,
            )
        ),
        "average_precision": float(
            average_precision_score(
                y_true,
                scores,
            )
        ),
        "precision_at_20":
            precision_at_k(
                y_true,
                scores,
                20,
            ),
        "precision_at_50":
            precision_at_k(
                y_true,
                scores,
                50,
            ),
        "precision_at_100":
            precision_at_k(
                y_true,
                scores,
                100,
            ),
        "lift_at_20":
            lift_at_k(
                y_true,
                scores,
                20,
            ),
        "lift_at_50":
            lift_at_k(
                y_true,
                scores,
                50,
            ),
        "lift_at_100":
            lift_at_k(
                y_true,
                scores,
                100,
            ),
    }


def make_models():
    return {
        "Logistic Regression": Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                ),
            ),
            (
                "scale",
                StandardScaler(),
            ),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    random_state=RANDOM_SEED,
                ),
            ),
        ]),

        "Random Forest": Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                ),
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=400,
                    min_samples_leaf=5,
                    class_weight="balanced_subsample",
                    random_state=RANDOM_SEED,
                    n_jobs=-1,
                ),
            ),
        ]),

        "HistGradientBoosting": Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                ),
            ),
            (
                "model",
                HistGradientBoostingClassifier(
                    max_iter=200,
                    learning_rate=0.05,
                    max_leaf_nodes=15,
                    l2_regularization=1.0,
                    random_state=RANDOM_SEED,
                ),
            ),
        ]),
    }


def fit_baseline_params(train_df):
    return {
        "imp_change_median": float(
            train_df[
                "imp_change_recent_vs_middle"
            ].median()
        ),

        "pos_change_median": float(
            train_df[
                "pos_change_recent_vs_middle"
            ].median()
        ),

        "visibility_max": max(
            1e-9,
            float(
                np.log1p(
                    train_df[
                        "imp_recent30"
                    ]
                ).max()
            ),
        ),
    }


def baseline_score(df, params):
    imp_change = (
        df[
            "imp_change_recent_vs_middle"
        ]
        .fillna(
            params[
                "imp_change_median"
            ]
        )
    )

    pos_change = (
        df[
            "pos_change_recent_vs_middle"
        ]
        .fillna(
            params[
                "pos_change_median"
            ]
        )
    )

    weakening = np.maximum(
        0,
        -imp_change,
    )

    worsening_pos = np.maximum(
        0,
        pos_change,
    )

    visibility = np.log1p(
        df["imp_recent30"]
    )

    return (
        0.50 * weakening
        + 0.25
        * np.log1p(
            worsening_pos
        )
        + 0.25
        * visibility
        / params[
            "visibility_max"
        ]
    ).to_numpy()


gkf = GroupKFold(
    n_splits=5
)

cv_rows = []


for fold, (tr, va) in enumerate(
    gkf.split(
        X,
        y,
        groups,
    ),
    start=1,
):

    yv = (
        y.iloc[va]
        .to_numpy()
    )


    fold_baseline_params = (
        fit_baseline_params(
            frame.iloc[tr]
        )
    )


    baseline_scores = (
        baseline_score(
            frame.iloc[va],
            fold_baseline_params,
        )
    )


    row = metric_row(
        "Transparent baseline",
        yv,
        baseline_scores,
    )

    row["fold"] = fold

    cv_rows.append(row)


    for name, model in (
        make_models().items()
    ):

        model.fit(
            X.iloc[tr],
            y.iloc[tr],
        )


        scores = (
            model.predict_proba(
                X.iloc[va]
            )[:, 1]
        )


        row = metric_row(
            name,
            yv,
            scores,
        )

        row["fold"] = fold

        cv_rows.append(row)


cv_results = pd.DataFrame(
    cv_rows
)


cv_summary = (
    cv_results
    .groupby("method")
    .agg(
        folds=(
            "fold",
            "count",
        ),

        base_rate_mean=(
            "base_rate",
            "mean",
        ),

        ap_mean=(
            "average_precision",
            "mean",
        ),

        ap_std=(
            "average_precision",
            "std",
        ),

        p20_mean=(
            "precision_at_20",
            "mean",
        ),

        p20_std=(
            "precision_at_20",
            "std",
        ),

        p50_mean=(
            "precision_at_50",
            "mean",
        ),

        p50_std=(
            "precision_at_50",
            "std",
        ),

        p100_mean=(
            "precision_at_100",
            "mean",
        ),

        p100_std=(
            "precision_at_100",
            "std",
        ),

        lift20_mean=(
            "lift_at_20",
            "mean",
        ),

        lift50_mean=(
            "lift_at_50",
            "mean",
        ),

        lift100_mean=(
            "lift_at_100",
            "mean",
        ),
    )
    .reset_index()
)


print(
    "Grouped cross-validation summary:"
)


display(
    cv_summary.style.format({
        "base_rate_mean": "{:.2%}",

        "ap_mean": "{:.4f}",
        "ap_std": "{:.4f}",

        "p20_mean": "{:.2%}",
        "p20_std": "{:.2%}",

        "p50_mean": "{:.2%}",
        "p50_std": "{:.2%}",

        "p100_mean": "{:.2%}",
        "p100_std": "{:.2%}",

        "lift20_mean": "{:.2f}x",
        "lift50_mean": "{:.2f}x",
        "lift100_mean": "{:.2f}x",
    })
)


learned = (
    cv_summary[
        cv_summary["method"]
        != "Transparent baseline"
    ]
    .copy()
)


# Official model selection:
# use only the predeclared mean grouped-CV
# Average Precision criterion.
SELECTED_MODEL = (
    learned
    .sort_values(
        "ap_mean",
        ascending=False,
    )
    .iloc[0]["method"]
)


# These are diagnostics only.
# They do not affect SELECTED_MODEL.
P20_WINNER = (
    learned
    .sort_values(
        "p20_mean",
        ascending=False,
    )
    .iloc[0]["method"]
)


P50_WINNER = (
    learned
    .sort_values(
        "p50_mean",
        ascending=False,
    )
    .iloc[0]["method"]
)


baseline_cv = (
    cv_summary.loc[
        cv_summary["method"]
        == "Transparent baseline"
    ]
    .iloc[0]
)


selected_cv = (
    cv_summary.loc[
        cv_summary["method"]
        == SELECTED_MODEL
    ]
    .iloc[0]
)


OPERATIONALLY_USEFUL = (
    selected_cv["p50_mean"]
    > selected_cv[
        "base_rate_mean"
    ]
    and
    selected_cv["p50_mean"]
    > baseline_cv[
        "p50_mean"
    ]
)


rf_cv = (
    cv_summary.loc[
        cv_summary["method"]
        == "Random Forest"
    ]
    .iloc[0]
)


hgb_cv = (
    cv_summary.loc[
        cv_summary["method"]
        == "HistGradientBoosting"
    ]
    .iloc[0]
)


selection_sensitivity = (
    pd.DataFrame([
        {
            "criterion":
                "Mean Average Precision",

            "winner":
                SELECTED_MODEL,

            "random_forest":
                rf_cv["ap_mean"],

            "hist_gradient_boosting":
                hgb_cv["ap_mean"],

            "difference_rf_minus_hgb":
                (
                    rf_cv["ap_mean"]
                    - hgb_cv["ap_mean"]
                ),
        },

        {
            "criterion":
                "Mean Precision@20",

            "winner":
                P20_WINNER,

            "random_forest":
                rf_cv["p20_mean"],

            "hist_gradient_boosting":
                hgb_cv["p20_mean"],

            "difference_rf_minus_hgb":
                (
                    rf_cv["p20_mean"]
                    - hgb_cv["p20_mean"]
                ),
        },

        {
            "criterion":
                "Mean Precision@50",

            "winner":
                P50_WINNER,

            "random_forest":
                rf_cv["p50_mean"],

            "hist_gradient_boosting":
                hgb_cv["p50_mean"],

            "difference_rf_minus_hgb":
                (
                    rf_cv["p50_mean"]
                    - hgb_cv["p50_mean"]
                ),
        },
    ])
)


print(
    "\nModel-selection sensitivity:"
)


display(
    selection_sensitivity.style.format({
        "random_forest":
            "{:.4f}",

        "hist_gradient_boosting":
            "{:.4f}",

        "difference_rf_minus_hgb":
            "{:+.4f}",
    })
)


fold_diagnostics = (
    cv_results[
        cv_results[
            "method"
        ].isin([
            SELECTED_MODEL,
            "HistGradientBoosting",
        ])
    ][[
        "fold",
        "method",
        "base_rate",
        "average_precision",
        "precision_at_20",
        "precision_at_50",
        "precision_at_100",
        "lift_at_20",
        "lift_at_50",
        "lift_at_100",
    ]]
    .sort_values([
        "fold",
        "method",
    ])
    .reset_index(
        drop=True
    )
)


print(
    "\nFold-level diagnostics:"
)


display(
    fold_diagnostics.style.format({
        "base_rate":
            "{:.2%}",

        "average_precision":
            "{:.4f}",

        "precision_at_20":
            "{:.2%}",

        "precision_at_50":
            "{:.2%}",

        "precision_at_100":
            "{:.2%}",

        "lift_at_20":
            "{:.2f}x",

        "lift_at_50":
            "{:.2f}x",

        "lift_at_100":
            "{:.2f}x",
    })
)


gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=RANDOM_SEED,
)


tr_g, te_g = next(
    gss.split(
        X,
        y,
        groups,
    )
)


train_clients = set(
    groups.iloc[tr_g]
)


test_clients = set(
    groups.iloc[te_g]
)


assert (
    train_clients
    .isdisjoint(
        test_clients
    )
)


test_frame = (
    frame.iloc[te_g]
    .copy()
)


yt = (
    y.iloc[te_g]
    .to_numpy()
)


holdout_baseline_params = (
    fit_baseline_params(
        frame.iloc[tr_g]
    )
)


holdout_rows = [
    metric_row(
        "Transparent baseline",
        yt,
        baseline_score(
            test_frame,
            holdout_baseline_params,
        ),
    )
]


fitted_models = {}
holdout_scores = {}


for name, model in (
    make_models().items()
):

    model.fit(
        X.iloc[tr_g],
        y.iloc[tr_g],
    )


    scores = (
        model.predict_proba(
            X.iloc[te_g]
        )[:, 1]
    )


    fitted_models[
        name
    ] = model


    holdout_scores[
        name
    ] = scores


    holdout_rows.append(
        metric_row(
            name,
            yt,
            scores,
        )
    )


comparison = pd.DataFrame(
    holdout_rows
)


print(
    "\nFixed grouped-holdout comparison:"
)


display(
    comparison.style.format({
        "base_rate":
            "{:.2%}",

        "roc_auc":
            "{:.3f}",

        "average_precision":
            "{:.4f}",

        "precision_at_20":
            "{:.2%}",

        "precision_at_50":
            "{:.2%}",

        "precision_at_100":
            "{:.2%}",

        "lift_at_20":
            "{:.2f}x",

        "lift_at_50":
            "{:.2f}x",

        "lift_at_100":
            "{:.2f}x",
    })
)


selected_model = (
    fitted_models[
        SELECTED_MODEL
    ]
)


prob = (
    holdout_scores[
        SELECTED_MODEL
    ]
)


print(
    "\nSelected learned model:",
    SELECTED_MODEL,
)


print(
    "Official selection criterion:",
    "highest mean grouped-CV Average Precision",
)


print(
    "Operational usefulness check passed:",
    OPERATIONALLY_USEFUL,
)


print(
    "P@20 diagnostic winner:",
    P20_WINNER,
)


print(
    "P@50 diagnostic winner:",
    P50_WINNER,
)


print(
    "Grouped holdout rows:",
    f"{len(te_g):,}",
)


print(
    "Held-out clients:",
    len(test_clients),
)


print(
    "Train/test client overlap:",
    len(
        train_clients
        & test_clients
    ),
)

Grouped cross-validation summary:


,method,folds,base_rate_mean,ap_mean,ap_std,p20_mean,p20_std,p50_mean,p50_std,p100_mean,p100_std,lift20_mean,lift50_mean,lift100_mean
0,HistGradientBoosting,5,30.04%,0.3575,0.1533,32.00%,24.90%,35.20%,23.69%,37.80%,19.98%,1.03x,1.17x,1.34x
1,Logistic Regression,5,30.04%,0.3171,0.1542,33.00%,13.51%,30.40%,9.32%,28.60%,10.45%,1.27x,1.13x,1.05x
2,Random Forest,5,30.04%,0.3616,0.1585,39.00%,21.62%,36.80%,18.85%,39.80%,19.23%,1.36x,1.30x,1.43x
3,Transparent baseline,5,30.04%,0.3476,0.2658,32.00%,39.31%,34.40%,37.67%,34.40%,36.46%,0.85x,0.95x,0.97x



Model-selection sensitivity:


,criterion,winner,random_forest,hist_gradient_boosting,difference_rf_minus_hgb
0,Mean Average Precision,Random Forest,0.3616,0.3575,+0.0042
1,Mean Precision@20,Random Forest,0.3900,0.3200,+0.0700
2,Mean Precision@50,Random Forest,0.3680,0.3520,+0.0160



Fold-level diagnostics:


,fold,method,base_rate,average_precision,precision_at_20,precision_at_50,precision_at_100,lift_at_20,lift_at_50,lift_at_100
0,1,HistGradientBoosting,26.68%,0.3786,40.00%,40.00%,38.00%,1.50x,1.50x,1.42x
1,1,Random Forest,26.68%,0.3805,45.00%,36.00%,42.00%,1.69x,1.35x,1.57x
2,2,HistGradientBoosting,21.45%,0.2804,10.00%,20.00%,30.00%,0.47x,0.93x,1.40x
3,2,Random Forest,21.45%,0.2800,35.00%,30.00%,36.00%,1.63x,1.40x,1.68x
4,3,HistGradientBoosting,18.88%,0.2908,30.00%,34.00%,42.00%,1.59x,1.80x,2.22x
5,3,Random Forest,18.88%,0.2952,35.00%,38.00%,41.00%,1.85x,2.01x,2.17x
6,4,HistGradientBoosting,20.38%,0.2243,10.00%,10.00%,12.00%,0.49x,0.49x,0.59x
7,4,Random Forest,20.38%,0.2253,10.00%,14.00%,13.00%,0.49x,0.69x,0.64x
8,5,HistGradientBoosting,62.80%,0.6133,70.00%,72.00%,67.00%,1.11x,1.15x,1.07x
9,5,Random Forest,62.80%,0.6272,70.00%,66.00%,67.00%,1.11x,1.05x,1.07x



Fixed grouped-holdout comparison:


,method,base_rate,roc_auc,average_precision,precision_at_20,precision_at_50,precision_at_100,lift_at_20,lift_at_50,lift_at_100
0,Transparent baseline,24.02%,0.537,0.2631,10.00%,20.00%,24.00%,0.42x,0.83x,1.00x
1,Logistic Regression,24.02%,0.559,0.2821,25.00%,32.00%,18.00%,1.04x,1.33x,0.75x
2,Random Forest,24.02%,0.620,0.3207,40.00%,40.00%,37.00%,1.67x,1.67x,1.54x
3,HistGradientBoosting,24.02%,0.619,0.3210,35.00%,40.00%,35.00%,1.46x,1.67x,1.46x



Selected learned model: Random Forest
Official selection criterion: highest mean grouped-CV Average Precision
Operational usefulness check passed: True
P@20 diagnostic winner: Random Forest
P@50 diagnostic winner: Random Forest
Grouped holdout rows: 9,092
Held-out clients: 6
Train/test client overlap: 0


### Same-run model comparison

>The Week 7 results separate two related purposes. **Client-grouped cross-validation is the main evidence used for model selection and robustness**, while the fixed client-grouped holdout provides a concrete same-run comparison for error analysis, client-level inspection, and construction of the reviewer-facing queue. All methods are evaluated under the same refined Week 7 population, historical feature definition, and leakage-safe timeline.

>I declared the model-selection rule before interpreting the final comparison: **select the learned model with the highest mean grouped-CV Average Precision**. Average Precision evaluates ranking quality across the precision-recall curve rather than at only one review depth. Precision@20, Precision@50, and their corresponding lift values remain important operational measures because they describe performance when review capacity is limited.

>The executed grouped-validation results support carrying **Random Forest forward as the selected Week 7 development model**. It achieved the highest mean grouped-CV Average Precision at **0.3616**, compared with **0.3575** for HistGradientBoosting and **0.3476** for the transparent baseline. Random Forest therefore improves mean Average Precision over the transparent baseline, although the improvement remains modest.

>Random Forest also achieved the strongest mean top-K precision among the learned models at the two primary review capacities: **39.0% Precision@20** and **36.8% Precision@50**, compared with **32.0%** and **35.2%**, respectively, for HistGradientBoosting. Relative to the **30.04% mean fold decline base rate**, Random Forest achieved mean lift of **1.36× at 20**, **1.30× at 50**, and **1.43× at 100**.

>The model-selection result is therefore more consistent in this execution than in some earlier development runs: **Random Forest leads the predeclared Average-Precision criterion and also leads the learned alternatives at Precision@20 and Precision@50**. I still retain Average Precision as the official selection criterion rather than changing the rule based on the observed top-K results.

>The fold-level results remain an important limitation. Random Forest's grouped-CV Average Precision averaged **0.3616 with a standard deviation of 0.1585**, while Precision@50 averaged **36.8% with a standard deviation of 18.85 percentage points**. Performance therefore varies substantially depending on which clients are held out. I treat Random Forest as the **best-supported development model under the declared selection rule**, not as a universally reliable or production-ready predictor.

>On the fixed six-client grouped holdout, Random Forest achieved ROC-AUC **0.620**, Average Precision **0.3207**, Precision@20 **40%**, Precision@50 **40%**, and Precision@100 **37%**, against a **24.02% decline base rate**. This corresponds to lift of **1.67× at 20**, **1.67× at 50**, and **1.54× at 100**. HistGradientBoosting remained highly competitive, with ROC-AUC **0.619**, Average Precision **0.3210**, Precision@20 **35%**, Precision@50 **40%**, and Precision@100 **35%**.

>The fixed holdout therefore supports the same broad conclusion as grouped cross-validation: the nonlinear models provide useful top-of-queue enrichment, with Random Forest showing the strongest grouped-CV evidence under the declared selection rule. The fixed holdout is used for queue construction and concrete error analysis; it does **not** override the grouped-CV model-selection rule.

### Model-selection sensitivity

>The model-selection sensitivity check asks whether the preferred learned model would change if the ranking were judged by a different operational metric. In the current execution, **Random Forest leads all three reported grouped-CV criteria**: mean Average Precision (**0.3616 vs. 0.3575**), mean Precision@20 (**39.0% vs. 32.0%**), and mean Precision@50 (**36.8% vs. 35.2%**) relative to HistGradientBoosting.

>The advantage is strongest at Precision@20, where Random Forest leads by **7.0 percentage points**. Its lead is smaller at Precision@50 (**1.6 percentage points**) and mean Average Precision (**0.0042**).

>This consistency strengthens the case for carrying Random Forest forward, but it does **not** change the model-selection rule. Mean grouped-CV Average Precision remains the official criterion because it was declared before interpreting the final comparison. Precision@20 and Precision@50 remain operational diagnostics rather than post-hoc replacement criteria.

>The practical interpretation is therefore that **Random Forest is the best-supported learned model in this development execution across both the declared whole-ranking criterion and the primary fixed review capacities**. The margins remain modest enough that I do not interpret this as evidence that Random Forest is universally superior to other nonlinear models.

### Fold-level stability

>The grouped-CV figure shows that the selected model's performance is **not uniform across held-out client groups**. Random Forest's Average Precision and Precision@50 change substantially across the five folds, alongside differences in the contemporary decline base rate.

>The fifth fold is particularly different from the others, with a decline base rate of **62.80%**, compared with approximately **18.88%–26.68%** in the other four folds. This demonstrates why raw Precision@K should be interpreted together with the corresponding base rate or lift rather than in isolation.

>Random Forest's Precision@50 lift ranges from **0.69× to 2.01×** across the five grouped folds. Four folds are above `1.0×`, while one fold falls below prevalence. The model therefore shows useful enrichment across much of the grouped validation but does **not** transfer equally well to every held-out client composition.

>This variability is an important limitation. It supports using the ranking as **directional human-review decision support** and reinforces the need for an untouched later-period evaluation before making a stronger claim about generalization.

## 3. Human review + the no-go list

*What a reviewer must check before acting, and what should never be automated.*

### Human-review rules

Before acting on a recommendation, the reviewer should ask:

1. **Is there enough evidence?** Even though the Week 7 eligibility rules require minimum historical visibility, tracking coverage, and observed page history, some recommendations still have stronger supporting evidence than others. Pages with weaker evidence should be reviewed more cautiously.

2. **Is the historical pattern meaningful and persistent?** The Week 7 model uses multiple historical blocks so the reviewer can distinguish recent level from earlier movement. A short-lived fluctuation or isolated change should not automatically trigger editorial work.

3. **Could something else explain the apparent risk?** Seasonality, changing search demand, consolidation, site-wide movement, SERP changes, competing pages, or client-specific events may explain the observed pattern even when the model assigns a high risk score.

4. **Does the page still satisfy its intended search need?** The model observes historical search behavior but cannot inspect the page's actual content, query intent, business purpose, or whether another page has become the more appropriate search result.

5. **Would editing create protection risk?** A page with substantial historical visibility or strong ranking value may have more to lose from an unnecessary intervention. High-priority pages should therefore be diagnosed before major edits are made.

6. **What do the reason codes actually say?** Codes such as `historical_position_worsening`, `weak_click_capture`, `meaningful_visibility`, or `limited_evidence` describe observable evidence behind the recommendation. They should guide what the reviewer investigates, not be treated as diagnoses.

7. **What was the final human decision?** Record whether the recommendation was accepted for action, monitored, or dismissed, together with a short reason. This creates feedback that can later show whether the queue is actually useful to reviewers.

### What the ranking errors tell me

>The validation results tell me whether the ranking has useful signal overall, but they do not show what the model gets wrong. I therefore inspect **high-scored actual negatives** and **low-scored actual positives** from the fixed client-grouped development holdout.

>A **high-scored actual negative** is a page that the model places near the top of the review queue even though it does not later meet the decline definition. These recommendations can consume limited review capacity and show that historical patterns associated with elevated model risk do not always lead to a later decline.

>A **low-scored actual positive** shows the opposite limitation: the page later meets the decline definition even though the model assigns it relatively low priority. These cases matter because a genuinely declining page could remain too far down the queue to receive early review.

>I keep these examples pseudonymized and do not assign a specific cause to an individual error. The Week 7 model has a richer historical view than the earlier three-feature model, but it still cannot observe everything that affects search performance. Seasonality, search-intent changes, competing pages, content changes, query mix, SERP changes, and client-specific events can still produce outcomes that the historical search features do not explain.

>What I want from these examples is narrower: **do the ranking mistakes reveal where the refined model and its reason codes are still too limited for autonomous decisions?** The errors help interpret the grouped-validation results and reinforce why the queue should prioritize investigation rather than diagnose individual pages.

### What should NOT be automated

I would **not** allow this model or playbook to automatically:

- publish, rewrite, or substantially edit content;
- refresh a page solely because its model score is high;
- delete or prune a page;
- merge pages or create redirects;
- change titles, metadata, canonicals, or other search-facing settings;
- declare that a page declined because it was stale, thin, poorly optimized, or affected by cannibalization;
- treat weak historical CTR as proof that the title or meta description is the problem;
- treat worsening historical position as proof that the page itself caused the ranking loss;
- treat a freshness-related observation as proof that refreshing the page will recover visibility;
- treat a high model score or reason code as proof that any specific editorial intervention will improve traffic;
- force pages that fail the Week 7 eligibility requirements into the ranked queue.

>The safe automation boundary is narrower: the system may **recompute leakage-safe historical features, apply the documented eligibility rules, score eligible pseudonymized pages, rank them, attach evidence-based reason codes, and prepare a review queue**. These steps organize evidence and reduce manual triage work; they do not replace editorial judgment.

>The final workflow remains: **prioritize → inspect → diagnose → decide**. A human reviewer remains responsible for deciding whether the appropriate response is to act, protect, monitor, consolidate, or make no change.

In [24]:
eval_queue = test_frame[[
    "client_hash_id",
    "content_hash_id",
    "imp_90d",
    "imp_recent30",
    "clk_recent30",
    "ctr_recent30",
    "pos_recent30",
    "imp_change_recent_vs_middle",
    "imp_change_middle_vs_early",
    "pos_change_recent_vs_middle",
    "days_with_impressions_90",
    "imp_cv_90",
    "is_declining",
]].copy()


eval_queue = eval_queue.rename(
    columns={
        "is_declining": "observed_holdout_outcome"
    }
)

eval_queue["model_score"] = prob


eval_queue = (
    eval_queue
    .sort_values(
        ["model_score", "imp_recent30"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)


eval_queue.insert(
    0,
    "rank",
    np.arange(1, len(eval_queue) + 1),
)


eval_queue["model_percentile"] = (
    1
    - (eval_queue["rank"] - 1)
    / len(eval_queue)
)


queue = eval_queue.drop(
    columns=["observed_holdout_outcome"]
).copy()


def reason_codes(r):
    codes = []

    if r["rank"] <= 50:
        codes.append("top50_model_rank")

    if r["model_percentile"] >= 0.90:
        codes.append("high_model_rank")

    if r["imp_recent30"] >= 1000:
        codes.append("meaningful_visibility")

    if (
        pd.notna(r["pos_recent30"])
        and 0 < r["pos_recent30"] <= 10
    ):
        codes.append("page_one_exposure")

    if (
        pd.notna(r["imp_change_middle_vs_early"])
        and r["imp_change_middle_vs_early"] < -0.10
    ):
        codes.append(
            "earlier_impression_weakening"
        )

    if (
        pd.notna(r["pos_change_recent_vs_middle"])
        and r["pos_change_recent_vs_middle"] > 2
    ):
        codes.append(
            "historical_position_worsening"
        )

    if (
        r["imp_recent30"] >= 500
        and pd.notna(r["pos_recent30"])
        and 0 < r["pos_recent30"] <= 20
        and pd.notna(r["ctr_recent30"])
        and r["ctr_recent30"] < 0.005
    ):
        codes.append("weak_click_capture")

    if (
        r["imp_recent30"] < 250
        or r["days_with_impressions_90"] < 45
    ):
        codes.append("limited_evidence")

    return (
        ";".join(codes)
        if codes
        else "model_rank_only"
    )


def suggested_action(r):
    codes = set(
        r["reason_codes"].split(";")
    )

    if "limited_evidence" in codes:
        return "monitor"

    if (
        "page_one_exposure" in codes
        and "high_model_rank" in codes
    ):
        return "protect_and_diagnose"

    if (
        "weak_click_capture" in codes
        and "high_model_rank" in codes
    ):
        return "review_click_capture"

    if (
        "top50_model_rank" in codes
        and "meaningful_visibility" in codes
    ):
        return "review_now"

    if "high_model_rank" in codes:
        return "diagnose"

    return "standard_review"


def confidence_note(r):
    if "limited_evidence" in r["reason_codes"]:
        return (
            "limited historical evidence; "
            "monitor before acting"
        )

    if r["rank"] <= 50:
        return (
            "top-capacity candidate; "
            "model rank is not a causal diagnosis"
        )

    return "ranking context only"


queue["reason_codes"] = queue.apply(
    reason_codes,
    axis=1,
)

queue["suggested_action"] = queue.apply(
    suggested_action,
    axis=1,
)

queue["confidence_note"] = queue.apply(
    confidence_note,
    axis=1,
)

queue["human_review_required"] = True


FORBIDDEN_QUEUE_COLUMNS = {
    "imp_outcome30",
    "is_declining",
    "observed_holdout_outcome",
    "trend_direction",
    "trend_pct",
}


assert not FORBIDDEN_QUEUE_COLUMNS.intersection(
    queue.columns
)


display(queue.head(20))


print("Top 20 action mix:")

display(
    queue
    .head(20)["suggested_action"]
    .value_counts()
    .rename("pages")
    .to_frame()
)


print("Top 50 action mix:")

display(
    queue
    .head(50)["suggested_action"]
    .value_counts()
    .rename("pages")
    .to_frame()
)


errors = eval_queue.copy()


print("Three highest-scored actual negatives:")

display(
    errors[
        errors["observed_holdout_outcome"] == 0
    ]
    .sort_values(
        "model_score",
        ascending=False,
    )
    .head(3)
)


print("Three lowest-scored actual positives:")

display(
    errors[
        errors["observed_holdout_outcome"] == 1
    ]
    .sort_values(
        "model_score",
        ascending=True,
    )
    .head(3)
)


client_rows = []


for client_id, g in errors.groupby(
    "client_hash_id"
):

    if (
        g["observed_holdout_outcome"]
        .nunique()
        < 2
    ):
        continue

    yy = (
        g["observed_holdout_outcome"]
        .to_numpy()
    )

    ss = (
        g["model_score"]
        .to_numpy()
    )

    client_rows.append({
        "client_hash_id": client_id,
        "n": len(g),
        "base_rate": yy.mean(),
        "average_precision": (
            average_precision_score(
                yy,
                ss,
            )
        ),
        "roc_auc": (
            roc_auc_score(
                yy,
                ss,
            )
        ),
        "precision_at_20": (
            precision_at_k(
                yy,
                ss,
                min(20, len(g)),
            )
        ),
    })


client_metrics = pd.DataFrame(
    client_rows
)


if not client_metrics.empty:
    client_metrics = (
        client_metrics
        .sort_values(
            "average_precision",
            ascending=False,
        )
        .reset_index(drop=True)
    )


print("Held-out client diagnostics:")


display(
    client_metrics.style.format({
        "base_rate": "{:.2%}",
        "average_precision": "{:.3f}",
        "roc_auc": "{:.3f}",
        "precision_at_20": "{:.2%}",
    })
)


selected_estimator = (
    selected_model.named_steps["model"]
    if (
        isinstance(
            selected_model,
            Pipeline,
        )
        and "model"
        in selected_model.named_steps
    )
    else selected_model
)


if hasattr(
    selected_estimator,
    "feature_importances_",
):

    importance = pd.Series(
        selected_estimator.feature_importances_,
        index=FEATURE_COLS,
    ).sort_values(
        ascending=False
    )

    print(
        "Selected-model feature importances:"
    )

    display(
        importance
        .rename("importance")
        .to_frame()
    )

else:

    importance = pd.Series(
        dtype=float
    )

    print(
        "Selected model does not expose "
        "impurity-based feature_importances_."
    )

,rank,client_hash_id,content_hash_id,imp_90d,imp_recent30,clk_recent30,ctr_recent30,pos_recent30,imp_change_recent_vs_middle,imp_change_middle_vs_early,pos_change_recent_vs_middle,days_with_impressions_90,imp_cv_90,model_score,model_percentile,reason_codes,suggested_action,confidence_note,human_review_required
0,1,client_62f4a7e64f5e0096,content_1b52535d52029033,500.0,154.0,0.0,0.000000,16.074493,0.006536,-0.207254,4.253114,75,0.768554,0.960260,1.00000,top50_model_rank;high_model_rank;earlier_impre...,monitor,limited historical evidence; monitor before ac...,True
1,2,client_73cda7b4e4f265ea,content_2e93e495f6d84775,719.0,218.0,0.0,0.000000,48.707945,0.033175,-0.272414,7.222897,79,0.998308,0.958449,0.99989,top50_model_rank;high_model_rank;earlier_impre...,monitor,limited historical evidence; monitor before ac...,True
2,3,client_62f4a7e64f5e0096,content_232c91ba3b38b443,400.0,111.0,0.0,0.000000,14.072956,-0.112000,-0.237805,4.520247,74,0.604662,0.955599,0.99978,top50_model_rank;high_model_rank;earlier_impre...,monitor,limited historical evidence; monitor before ac...,True
3,4,client_62f4a7e64f5e0096,content_1aee3c2049ed782d,678.0,204.0,0.0,0.000000,45.591728,0.152542,-0.404040,16.685853,79,1.014008,0.942626,0.99967,top50_model_rank;high_model_rank;earlier_impre...,monitor,limited historical evidence; monitor before ac...,True
4,5,client_73cda7b4e4f265ea,content_9853f184bdce991f,2353.0,727.0,0.0,0.000000,29.848407,0.000000,-0.191324,-4.139716,80,0.637650,0.940024,0.99956,top50_model_rank;high_model_rank;earlier_impre...,diagnose,top-capacity candidate; model rank is not a ca...,True
5,6,client_62f4a7e64f5e0096,content_bf945843c6f7dff6,347.0,131.0,0.0,0.000000,25.836260,0.056452,0.347826,11.335917,73,0.879208,0.936482,0.99945,top50_model_rank;high_model_rank;historical_po...,monitor,limited historical evidence; monitor before ac...,True
6,7,client_62f4a7e64f5e0096,content_5c6454bb5ce22c95,2329.0,686.0,0.0,0.000000,15.346068,-0.195780,0.079747,0.751189,80,0.449494,0.923490,0.99934,top50_model_rank;high_model_rank;weak_click_ca...,review_click_capture,top-capacity candidate; model rank is not a ca...,True
7,8,client_62f4a7e64f5e0096,content_6867553a2768bc84,424.0,135.0,0.0,0.000000,6.789444,-0.123377,0.140741,2.863530,76,0.496966,0.918047,0.99923,top50_model_rank;high_model_rank;page_one_expo...,monitor,limited historical evidence; monitor before ac...,True
8,9,client_62f4a7e64f5e0096,content_fe224efec022f1a8,2758.0,842.0,3.0,0.003563,5.770008,-0.177734,0.147982,1.660562,80,0.366964,0.915337,0.99912,top50_model_rank;high_model_rank;page_one_expo...,protect_and_diagnose,top-capacity candidate; model rank is not a ca...,True
9,10,client_62f4a7e64f5e0096,content_f30c9b3791e5442d,582.0,190.0,0.0,0.000000,1.955060,-0.124424,0.240000,-0.349431,73,0.685405,0.907758,0.99901,top50_model_rank;high_model_rank;page_one_expo...,monitor,limited historical evidence; monitor before ac...,True


Top 20 action mix:


,pages
suggested_action,
monitor,12
protect_and_diagnose,5
diagnose,2
review_click_capture,1


Top 50 action mix:


,pages
suggested_action,
monitor,24
protect_and_diagnose,22
diagnose,2
review_click_capture,2


...

,rank,client_hash_id,content_hash_id,imp_90d,imp_recent30,clk_recent30,ctr_recent30,pos_recent30,imp_change_recent_vs_middle,imp_change_middle_vs_early,pos_change_recent_vs_middle,days_with_impressions_90,imp_cv_90,observed_holdout_outcome,model_score,model_percentile
9091,9092,client_62f4a7e64f5e0096,content_ea85810f7e5cc17e,3465.0,1325.0,10.0,0.007547,7.127758,0.045777,0.451317,-1.168170,90,0.375941,1,0.037516,0.00011
9079,9080,client_73cda7b4e4f265ea,content_e66c97dfa637e475,1982.0,719.0,6.0,0.008345,5.341358,0.076347,0.122689,-0.205356,90,0.400401,1,0.053911,0.00143
9074,9075,client_62f4a7e64f5e0096,content_d85df0f6adeec6da,3677.0,1394.0,11.0,0.007891,3.586806,-0.009240,0.606164,-0.489460,90,0.304112,1,0.058773,0.00198


Held-out client diagnostics:


,client_hash_id,n,base_rate,average_precision,roc_auc,precision_at_20
0,client_0797ff3a1fc9a6a5,4,75.00%,1.000,1.000,75.00%
1,client_cd12bcfd98942aa1,7,28.57%,0.667,0.600,28.57%
2,client_b10cb2997d0c7c86,26,34.62%,0.545,0.765,45.00%
3,client_62f4a7e64f5e0096,4464,26.68%,0.370,0.647,40.00%
4,client_73cda7b4e4f265ea,4392,21.45%,0.265,0.587,15.00%
5,client_400c21c81c8b46ef,199,18.59%,0.217,0.544,30.00%


Selected-model feature importances:


,importance
days_with_impressions_90,0.235648
imp_cv_90,0.107240
imp_change_middle_vs_early,0.086139
imp_early30,0.075952
pos_change_recent_vs_middle,0.075923
pos_recent30,0.074480
imp_recent30,0.066740
imp_middle30,0.063092
imp_change_recent_vs_middle,0.062851
ctr_change_recent_vs_middle,0.044367


## 4. Monitoring / retrain triggers

*What would tell me the recommendations have gone stale?*

>Monitoring stays light because this is a **research prototype, not a production system**. The main question is whether the refined Week 7 ranking continues to help reviewers prioritize eligible pages better than the contemporary decline base rate and transparent baseline. Monitoring should therefore focus on ranking usefulness, population changes, and whether the historical evidence entering the model still resembles the development setting.

For each new evaluation window, I would track:

- Precision@20;
- Precision@50;
- Precision@100;
- **Lift@20, Lift@50, and Lift@100 relative to the contemporary decline base rate**;
- Average Precision;
- ROC-AUC;
- current decline base rate;
- number of eligible pages and clients;
- percentage of pages passing the stability and evidence requirements;
- model-score distribution;
- historical feature distributions;
- missingness and historical coverage;
- **fold- or client-level performance dispersion**;
- reason-code and action mix; and
- recorded reviewer decisions such as `act`, `monitor`, or `dismiss`.

I would **revalidate** the model when:

- Precision@20 or Precision@50 moves close to the contemporary decline base rate;
- **top-K lift approaches `1.0x`, meaning the queue no longer concentrates later-declining pages above the contemporary base rate**;
- Average Precision no longer shows useful ranking improvement;
- the transparent baseline performs as well as or better than the learned model;
- performance varies substantially across clients or becomes concentrated in only a few client groups;
- the eligible-page population changes substantially;
- the stability, history-coverage, or minimum-impression rules select a meaningfully different population;
- impressions, clicks, position, movement, volatility, coverage, or model-score distributions change materially; or
- missingness or GSC tracking coverage changes enough that the historical feature set is no longer comparable with the development data.

I would **retrain and repeat the validation process** when:

- enough new labeled outcome data has accumulated to justify updating the model;
- the historical feature window or decline definition changes;
- the stability or eligibility definition changes materially;
- a new predictive feature is introduced;
- the underlying page or client population changes enough that the existing model is no longer representative; or
- repeated evaluation shows that the learned ranking has lost its advantage over the transparent baseline.

>Any change to the feature set or prediction timeline requires another **leakage audit** before an apparent improvement is trusted. I would also investigate an unusually strong top-K result rather than automatically treating it as evidence of a better model.

>I would **not retrain on a fixed schedule simply for the sake of retraining**. Before making a stronger generalization claim, I would freeze this development specification and evaluate it once on an untouched later time period. That later evaluation should test the frozen specification rather than become another tuning set.


In [25]:
sensitivity_rows = []

for min_imp in [100, 300, 500]:

    for width in [0.15, 0.20, 0.30]:

        ratio = np.where(
            features["imp_middle30"] > 0,
            features["imp_recent30"]
            / features["imp_middle30"],
            np.nan,
        )

        eligible = (
            (features["imp_recent30"] >= min_imp)
            & features["tracking_history_ok"]
            & (
                features["observed_history_days_90"]
                >= MIN_HISTORY_DAYS
            )
            & np.isfinite(ratio)
            & (ratio >= 1 - width)
            & (ratio <= 1 + width)
        )

        temp = features.loc[eligible]

        if len(temp) == 0:
            continue

        for drop in [0.15, 0.20, 0.30]:

            yy = (
                temp["imp_outcome30"]
                < (1 - drop)
                * temp["imp_recent30"]
            ).astype(int)

            sensitivity_rows.append({
                "min_recent_impressions": min_imp,
                "stability_band": (
                    f"±{int(width * 100)}%"
                ),
                "future_decline": (
                    f">{int(drop * 100)}% drop"
                ),
                "eligible_pages": len(temp),
                "clients": (
                    temp["client_hash_id"].nunique()
                ),
                "decline_base_rate": yy.mean(),
                "main_specification": (
                    min_imp
                    == MIN_RECENT_IMPRESSIONS
                    and np.isclose(
                        width,
                        1 - STABILITY_LOWER,
                    )
                    and np.isclose(
                        drop,
                        1 - DECLINE_RATIO,
                    )
                ),
            })


sensitivity = pd.DataFrame(sensitivity_rows)


sensitivity = (
    sensitivity
    .sort_values(
        [
            "main_specification",
            "min_recent_impressions",
            "stability_band",
            "future_decline",
        ],
        ascending=[
            False,
            True,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)


display(
    sensitivity.style.format({
        "decline_base_rate": "{:.2%}"
    })
)


monitoring_policy = {

    "headline_metrics": [
        "precision_at_20",
        "precision_at_50",
        "precision_at_100",
        "lift_at_20",
        "lift_at_50",
        "lift_at_100",
        "average_precision",
        "roc_auc",
    ],

    "compare_against": [
        "contemporary decline base rate",
        "transparent historical baseline",
    ],

    "check_per_client": True,

    "check_fold_or_client_dispersion": True,

    "revalidate_when_lift_approaches_one": True,

    "revalidate_on_population_shift": True,

    "re_audit_on_feature_window_or_label_change": True,

    "re_audit_on_new_feature": True,

    "investigate_implausibly_strong_results": True,

    "fixed_schedule_retraining_required": False,

    "sealed_future_test_required_for_stronger_generalization": True,

    "autonomous_editorial_actions_allowed": False,
}


monitoring_policy

,min_recent_impressions,stability_band,future_decline,eligible_pages,clients,decline_base_rate,main_specification
0,100,±20%,>20% drop,16183,24,28.58%,True
1,100,±15%,>15% drop,12392,24,32.84%,False
2,100,±15%,>20% drop,12392,24,28.31%,False
3,100,±15%,>30% drop,12392,24,20.38%,False
4,100,±20%,>15% drop,16183,24,33.23%,False
5,100,±20%,>30% drop,16183,24,20.73%,False
6,100,±30%,>15% drop,23091,25,33.52%,False
7,100,±30%,>20% drop,23091,25,28.89%,False
8,100,±30%,>30% drop,23091,25,21.23%,False
9,300,±15%,>15% drop,9277,19,31.78%,False


{'headline_metrics': ['precision_at_20',
  'precision_at_50',
  'precision_at_100',
  'lift_at_20',
  'lift_at_50',
  'lift_at_100',
  'average_precision',
  'roc_auc'],
 'compare_against': ['contemporary decline base rate',
  'transparent historical baseline'],
 'check_per_client': True,
 'check_fold_or_client_dispersion': True,
 'revalidate_when_lift_approaches_one': True,
 'revalidate_on_population_shift': True,
 're_audit_on_feature_window_or_label_change': True,
 're_audit_on_new_feature': True,
 'investigate_implausibly_strong_results': True,
 'fixed_schedule_retraining_required': False,
 'sealed_future_test_required_for_stronger_generalization': True,
 'autonomous_editorial_actions_allowed': False}

## 5. Exports for the paper

*The paper should be able to trace its recommendations and numbers back to this notebook.*

This notebook exports:

- `work/outputs/w07_ranked_action_queue.csv` — the **decision-time-safe human-review queue** containing pseudonymized page identifiers, model scores, review priority, reason codes, suggested actions, confidence/evidence notes, and human-review requirements. The later observed outcome is deliberately excluded.
- `work/outputs/w07_model_comparison.csv` — the fixed client-grouped holdout comparison used for concrete queue interpretation and error analysis.
- `work/outputs/w07_grouped_cv_results.csv` — the fold-level client-grouped cross-validation results used for model selection and robustness assessment.
- `work/outputs/w07_model_selection_sensitivity.csv` — a compact diagnostic showing which learned model leads under mean Average Precision, Precision@20, and Precision@50. This does not replace the predeclared Average-Precision selection rule.
- `work/outputs/w07_client_diagnostics.csv` — client-level evaluation results used to check whether performance is broadly distributed or concentrated in only a small number of held-out clients.
- `work/outputs/w07_playbook_metrics.json` — the reproducibility receipt containing the modeling specification, eligibility rules, feature set, selected model, validation design, model-selection sensitivity, headline metrics, and operational-usefulness check.
- `work/figures/w07_precision_at_k.png` — a reusable grouped cross-validation comparison of mean Precision@20, Precision@50, and Precision@100 across the transparent baseline and learned models, with the mean grouped-CV decline base rate shown for context.
- `work/figures/w07_grouped_cv_stability.png` — a fold-level visualization of the selected model's Average Precision, Precision@50, and contemporary decline base rate across held-out client groups.

>The exported artifacts serve different purposes. **Grouped cross-validation supports model selection and robustness**, while the fixed grouped holdout supports concrete error analysis, client-level inspection, and construction of the reviewer-facing queue. I keep these roles separate so that one favorable holdout result does not become the sole basis for selecting the development model.

### Final recommendation

>The executed Week 7 results support carrying **Random Forest forward as the selected development model under the predeclared mean grouped-CV Average Precision rule**. Random Forest achieved mean Average Precision of **0.3616**, compared with **0.3575** for HistGradientBoosting and **0.3476** for the transparent baseline.

>Random Forest also achieved mean Precision@20 of **39.0%** and Precision@50 of **36.8%**, compared with the **30.04% mean fold decline base rate**. This corresponds to mean lift of **1.36× at 20** and **1.30× at 50**. In this execution, Random Forest leads the learned alternatives on Average Precision, Precision@20, and Precision@50.

>The model-selection sensitivity analysis therefore supports the same model under all three reported grouped-CV criteria. I nevertheless retain **Average Precision as the official model-selection criterion** because it was declared before the final comparison. The top-K results provide operational support for that choice rather than replacing the selection rule.

>The grouped-CV variability remains an important limitation. Random Forest's mean Average Precision had a standard deviation of **0.1585**, while Precision@50 had a standard deviation of **18.85 percentage points**. One grouped fold also produced Precision@50 lift below `1.0×`, showing that the ranking does not transfer equally well to every held-out client composition.

>On the fixed six-client grouped holdout, Random Forest achieved ROC-AUC **0.620**, Average Precision **0.3207**, Precision@20 **40%**, Precision@50 **40%**, and lift of **1.67× at both 20 and 50**, against a **24.02% decline base rate**. HistGradientBoosting achieved slightly higher holdout Average Precision (**0.3210**) and the same Precision@50 (**40%**), which reinforces why the fixed holdout should be interpreted as a diagnostic rather than used to replace the grouped-CV selection rule.

>I therefore recommend using Random Forest as a **development-stage human-review prioritization model**, beginning with the number of pages the editorial team can realistically inspect. A high model score should determine **review order**, not the final editorial action.

>Refresh remains **one possible action, not the default action**. A surfaced page may instead require protection, diagnosis, monitoring, expansion, consolidation, or no change after human review.

>The conclusion carried into the paper is:

>**Historical search behavior provides measurable but variable value for prioritizing apparently stable pages for earlier human review. Random Forest is the best-supported development model under the predeclared Average-Precision criterion and also leads the primary grouped-CV top-K measures in the final execution, but substantial client-level variability still does not support autonomous editorial decisions or claims of universal predictive reliability.**

In [26]:
OUT = Path(
    "work/outputs"
)

FIG = Path(
    "work/figures"
)


OUT.mkdir(
    parents=True,
    exist_ok=True,
)


FIG.mkdir(
    parents=True,
    exist_ok=True,
)


queue_path = (
    OUT
    / "w07_ranked_action_queue.csv"
)


comparison_path = (
    OUT
    / "w07_model_comparison.csv"
)


cv_path = (
    OUT
    / "w07_grouped_cv_results.csv"
)


selection_path = (
    OUT
    / "w07_model_selection_sensitivity.csv"
)


client_path = (
    OUT
    / "w07_client_diagnostics.csv"
)


metrics_path = (
    OUT
    / "w07_playbook_metrics.json"
)


precision_fig_path = (
    FIG
    / "w07_precision_at_k.png"
)


stability_fig_path = (
    FIG
    / "w07_grouped_cv_stability.png"
)


queue.to_csv(
    queue_path,
    index=False,
)


comparison.to_csv(
    comparison_path,
    index=False,
)


cv_results.to_csv(
    cv_path,
    index=False,
)


selection_sensitivity.to_csv(
    selection_path,
    index=False,
)


client_metrics.to_csv(
    client_path,
    index=False,
)


# Precision@K comparison figure
precision_plot = (
    cv_summary[
        [
            "method",
            "p20_mean",
            "p50_mean",
            "p100_mean",
        ]
    ]
    .copy()
    .set_index("method")
)


ax = precision_plot.plot(
    kind="bar",
    figsize=(10, 5),
)


ax.axhline(
    float(
        cv_summary[
            "base_rate_mean"
        ].mean()
    ),
    linestyle="--",
    label="Mean grouped-CV base rate",
)


ax.set_ylabel(
    "Precision"
)


ax.set_title(
    "Week 7 grouped-CV Precision@K"
)


ax.legend(
    bbox_to_anchor=(
        1.02,
        1,
    ),
    loc="upper left",
)


plt.tight_layout()


plt.savefig(
    precision_fig_path,
    dpi=180,
    bbox_inches="tight",
)


plt.close()


# Fold-level stability figure
selected_fold = (
    cv_results[
        cv_results["method"]
        == SELECTED_MODEL
    ]
    .sort_values(
        "fold"
    )
)


fig, ax = plt.subplots(
    figsize=(8, 4.5)
)


ax.plot(
    selected_fold["fold"],
    selected_fold[
        "average_precision"
    ],
    marker="o",
    label="Average Precision",
)


ax.plot(
    selected_fold["fold"],
    selected_fold[
        "precision_at_50"
    ],
    marker="o",
    label="Precision@50",
)


ax.plot(
    selected_fold["fold"],
    selected_fold[
        "base_rate"
    ],
    linestyle="--",
    marker="o",
    label="Fold base rate",
)


ax.set_xlabel(
    "Grouped CV fold"
)


ax.set_ylabel(
    "Metric value"
)


ax.set_title(
    f"{SELECTED_MODEL}: "
    "performance across held-out client folds"
)


ax.set_xticks(
    selected_fold[
        "fold"
    ]
)


ax.set_ylim(
    0,
    min(
        1.0,
        max(
            selected_fold[
                [
                    "average_precision",
                    "precision_at_50",
                    "base_rate",
                ]
            ]
            .max()
            .max()
            * 1.15,
            0.50,
        ),
    ),
)


ax.legend()


ax.grid(
    axis="y",
    alpha=0.2,
)


plt.tight_layout()


fig.savefig(
    stability_fig_path,
    dpi=180,
    bbox_inches="tight",
)


plt.close(fig)


selected_holdout = (
    comparison.loc[
        comparison["method"]
        == SELECTED_MODEL
    ]
    .iloc[0]
)


baseline_holdout = (
    comparison.loc[
        comparison["method"]
        == "Transparent baseline"
    ]
    .iloc[0]
)


metrics_receipt = {
    "random_seed":
        RANDOM_SEED,

    "development_endpoint":
        str(
            DEVELOPMENT_END.date()
        ),

    "decision_date":
        str(
            DECISION_DATE.date()
        ),

    "historical_feature_start":
        str(
            HISTORICAL_FEATURE_START.date()
        ),

    "eligible_pages":
        int(
            len(frame)
        ),

    "eligible_clients":
        int(
            frame[
                "client_hash_id"
            ].nunique()
        ),

    "eligible_decline_base_rate":
        float(
            frame[
                "is_declining"
            ].mean()
        ),

    "feature_count":
        int(
            len(FEATURE_COLS)
        ),

    "feature_columns":
        FEATURE_COLS,

    "selection_rule": (
        "highest mean grouped-CV "
        "Average Precision among "
        "learned candidate models"
    ),

    "selection_rule_predeclared":
        True,

    "selected_model":
        SELECTED_MODEL,

    "operational_usefulness_rule": (
        "selected model mean grouped-CV "
        "Precision@50 must exceed both "
        "the mean grouped-CV decline "
        "base rate and the transparent "
        "baseline mean Precision@50"
    ),

    "operational_usefulness_passed":
        bool(
            OPERATIONALLY_USEFUL
        ),

    "alternative_metric_winners": {
        "precision_at_20":
            P20_WINNER,

        "precision_at_50":
            P50_WINNER,
    },

    "selected_cv": {
        "average_precision_mean":
            float(
                selected_cv[
                    "ap_mean"
                ]
            ),

        "average_precision_std":
            float(
                selected_cv[
                    "ap_std"
                ]
            ),

        "precision_at_20_mean":
            float(
                selected_cv[
                    "p20_mean"
                ]
            ),

        "precision_at_20_std":
            float(
                selected_cv[
                    "p20_std"
                ]
            ),

        "precision_at_50_mean":
            float(
                selected_cv[
                    "p50_mean"
                ]
            ),

        "precision_at_50_std":
            float(
                selected_cv[
                    "p50_std"
                ]
            ),

        "precision_at_100_mean":
            float(
                selected_cv[
                    "p100_mean"
                ]
            ),

        "precision_at_100_std":
            float(
                selected_cv[
                    "p100_std"
                ]
            ),

        "lift_at_20_mean":
            float(
                selected_cv[
                    "lift20_mean"
                ]
            ),

        "lift_at_50_mean":
            float(
                selected_cv[
                    "lift50_mean"
                ]
            ),

        "lift_at_100_mean":
            float(
                selected_cv[
                    "lift100_mean"
                ]
            ),

        "base_rate_mean":
            float(
                selected_cv[
                    "base_rate_mean"
                ]
            ),
    },

    "baseline_cv": {
        "average_precision_mean":
            float(
                baseline_cv[
                    "ap_mean"
                ]
            ),

        "precision_at_20_mean":
            float(
                baseline_cv[
                    "p20_mean"
                ]
            ),

        "precision_at_50_mean":
            float(
                baseline_cv[
                    "p50_mean"
                ]
            ),

        "precision_at_100_mean":
            float(
                baseline_cv[
                    "p100_mean"
                ]
            ),

        "lift_at_20_mean":
            float(
                baseline_cv[
                    "lift20_mean"
                ]
            ),

        "lift_at_50_mean":
            float(
                baseline_cv[
                    "lift50_mean"
                ]
            ),

        "lift_at_100_mean":
            float(
                baseline_cv[
                    "lift100_mean"
                ]
            ),

        "base_rate_mean":
            float(
                baseline_cv[
                    "base_rate_mean"
                ]
            ),
    },

    "selected_holdout": {
        "roc_auc":
            float(
                selected_holdout[
                    "roc_auc"
                ]
            ),

        "average_precision":
            float(
                selected_holdout[
                    "average_precision"
                ]
            ),

        "precision_at_20":
            float(
                selected_holdout[
                    "precision_at_20"
                ]
            ),

        "precision_at_50":
            float(
                selected_holdout[
                    "precision_at_50"
                ]
            ),

        "precision_at_100":
            float(
                selected_holdout[
                    "precision_at_100"
                ]
            ),

        "lift_at_20":
            float(
                selected_holdout[
                    "lift_at_20"
                ]
            ),

        "lift_at_50":
            float(
                selected_holdout[
                    "lift_at_50"
                ]
            ),

        "lift_at_100":
            float(
                selected_holdout[
                    "lift_at_100"
                ]
            ),

        "base_rate":
            float(
                selected_holdout[
                    "base_rate"
                ]
            ),
    },

    "baseline_holdout": {
        "roc_auc":
            float(
                baseline_holdout[
                    "roc_auc"
                ]
            ),

        "average_precision":
            float(
                baseline_holdout[
                    "average_precision"
                ]
            ),

        "precision_at_20":
            float(
                baseline_holdout[
                    "precision_at_20"
                ]
            ),

        "precision_at_50":
            float(
                baseline_holdout[
                    "precision_at_50"
                ]
            ),

        "precision_at_100":
            float(
                baseline_holdout[
                    "precision_at_100"
                ]
            ),

        "lift_at_20":
            float(
                baseline_holdout[
                    "lift_at_20"
                ]
            ),

        "lift_at_50":
            float(
                baseline_holdout[
                    "lift_at_50"
                ]
            ),

        "lift_at_100":
            float(
                baseline_holdout[
                    "lift_at_100"
                ]
            ),

        "base_rate":
            float(
                baseline_holdout[
                    "base_rate"
                ]
            ),
    },

    "model_selection_sensitivity":
        selection_sensitivity.to_dict(
            orient="records"
        ),

    "grouped_cv_folds":
        5,

    "fixed_holdout_rows":
        int(
            len(test_frame)
        ),

    "fixed_holdout_clients":
        int(
            len(test_clients)
        ),

    "train_test_client_overlap":
        int(
            len(
                train_clients
                & test_clients
            )
        ),

    "validation_scope": (
        "five-fold client-grouped "
        "development cross-validation "
        "for model selection and "
        "robustness, plus a fixed "
        "client-grouped development "
        "holdout for queue construction, "
        "error analysis, and client-level "
        "diagnostics; not a sealed "
        "future-month test"
    ),

    "preprocessing": (
        "median imputation is fitted "
        "inside each training fold or "
        "training partition; baseline "
        "medians and normalization "
        "parameters are also estimated "
        "from training clients only"
    ),

    "interpretation": (
        "Random Forest is selected under "
        "the predeclared mean grouped-CV "
        "Average Precision criterion. "
        "Precision@20 and Precision@50 "
        "winners are reported only as "
        "decision-sensitivity diagnostics "
        "and do not alter official model "
        "selection."
    ),

    "autonomous_editorial_actions_allowed":
        False,

    "sealed_future_test_required_for_stronger_generalization":
        True,
}


with open(
    metrics_path,
    "w",
) as f:

    json.dump(
        metrics_receipt,
        f,
        indent=2,
    )


print(
    "Wrote:",
    queue_path,
)

print(
    "Wrote:",
    comparison_path,
)

print(
    "Wrote:",
    cv_path,
)

print(
    "Wrote:",
    selection_path,
)

print(
    "Wrote:",
    client_path,
)

print(
    "Wrote:",
    metrics_path,
)

print(
    "Wrote:",
    precision_fig_path,
)

print(
    "Wrote:",
    stability_fig_path,
)


print(
    "\nSelected model:",
    SELECTED_MODEL,
)

print(
    "Selection rule:",
    metrics_receipt[
        "selection_rule"
    ],
)

print(
    "Operational usefulness passed:",
    OPERATIONALLY_USEFUL,
)

print(
    "P@20 diagnostic winner:",
    P20_WINNER,
)

print(
    "P@50 diagnostic winner:",
    P50_WINNER,
)

print(
    "Train/test client overlap:",
    metrics_receipt[
        "train_test_client_overlap"
    ],
)

Wrote: work/outputs/w07_ranked_action_queue.csv
Wrote: work/outputs/w07_model_comparison.csv
Wrote: work/outputs/w07_grouped_cv_results.csv
Wrote: work/outputs/w07_model_selection_sensitivity.csv
Wrote: work/outputs/w07_client_diagnostics.csv
Wrote: work/outputs/w07_playbook_metrics.json
Wrote: work/figures/w07_precision_at_k.png
Wrote: work/figures/w07_grouped_cv_stability.png

Selected model: Random Forest
Selection rule: highest mean grouped-CV Average Precision among learned candidate models
Operational usefulness passed: True
P@20 diagnostic winner: Random Forest
P@50 diagnostic winner: Random Forest
Train/test client overlap: 0


## Self-check

Before you submit, confirm each line honestly:

- [ - ] Every section above is filled — markdown thinking AND the code that backs it
- [ - ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ - ] No client names, URLs, or private queries anywhere
- [ - ] My claims use careful words: observed, measured, directional, decision-support
- [ - ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.